<div style='background:#0a0a0a;color:#f0c040;padding:16px;font-size:22px;border-radius:8px;text-align:center;font-weight:bold;'>
Phân loại trạng thái tấm pin năng lượng mặt trời<br/>
<span style='font-size:14px;color:#aaa'>Solar Panel Condition Classification — Clean / Dusty / Snow</span>
</div>
<div style='color:#888;text-align:center;margin-top:8px;font-size:13px'>
Models: EfficientNetB4 · ResNet50 · ViT-B/16 &nbsp;|&nbsp; PyTorch · Transfer Learning · 2-Phase Fine-tuning
</div>

## 1. Import thư viện

In [ ]:
import os
import copy
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import (
    efficientnet_b4, EfficientNet_B4_Weights,
    resnet50,        ResNet50_Weights,
    vit_b_16,        ViT_B_16_Weights,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

## 2. Cấu hình chung

In [ ]:
SEED     = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR   = 'data'       # thư mục chứa Clean / Dusty / Snow
MODELS_DIR = 'models'     # nơi lưu checkpoint .pt
IMG_SIZE   = 224
BATCH_SIZE = 32

NUM_EPOCHS_PHASE1 = 10    # freeze backbone, train head
NUM_EPOCHS_PHASE2 = 10    # unfreeze toàn bộ, fine-tune
LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-4

NUM_CLASSES = 3
CLASS_NAMES = ['Clean', 'Dusty', 'Snow']
# Số ảnh mỗi class — dùng để tính class weight bù imbalance
CLASS_COUNTS = [1493, 1069, 419]

os.makedirs(MODELS_DIR, exist_ok=True)
print(f'Device : {DEVICE}')
print(f'Data   : {os.path.abspath(DATA_DIR)}')
print(f'Models : {os.path.abspath(MODELS_DIR)}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
class_counts = {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in CLASS_NAMES}
total = sum(class_counts.values())

print('Phân phối dữ liệu:')
for cls, cnt in class_counts.items():
    print(f'  {cls:<8}: {cnt:>5}  ({cnt/total*100:.1f}%)')
print(f'  {"TOTAL":<8}: {total:>5}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#4CAF50', '#FF9800', '#2196F3']

axes[0].bar(class_counts.keys(), class_counts.values(), color=colors)
axes[0].set_title('Số lượng ảnh theo lớp'); axes[0].set_ylabel('Số ảnh')
for i, (_, cnt) in enumerate(class_counts.items()):
    axes[0].text(i, cnt + 10, str(cnt), ha='center', fontweight='bold')

axes[1].pie(class_counts.values(), labels=class_counts.keys(),
            colors=colors, autopct='%.1f%%', startangle=90)
axes[1].set_title('Tỷ lệ phân phối')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for row, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(DATA_DIR, cls)
    for col, fname in enumerate(random.sample(os.listdir(cls_dir), 4)):
        img = Image.open(os.path.join(cls_dir, fname)).resize((224, 224))
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls, fontsize=9)
        axes[row, col].axis('off')
plt.suptitle('Sample ảnh mỗi class', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Data Preparation — Augmentation & DataLoader

In [ ]:
# Chiến lược chia: 70% train | 15% val | 15% test (stratified)
# Augmentation mạnh cho train vì Snow class chỉ có 419 ảnh

class SolarPanelDataModule:

    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, data_dir, img_size=224, batch_size=32, seed=42):
        self.data_dir   = data_dir
        self.img_size   = img_size
        self.batch_size = batch_size
        self.seed       = seed
        self._build_transforms()

    def _build_transforms(self):
        sz = self.img_size
        self.train_tf = transforms.Compose([
            transforms.Resize((sz + 32, sz + 32)),
            transforms.RandomCrop(sz),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(degrees=20),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
            transforms.RandomGrayscale(p=0.05),
            transforms.ToTensor(),
            transforms.Normalize(self.MEAN, self.STD),
            transforms.RandomErasing(p=0.1),  # giả lập vật cản che khuất
        ])
        self.eval_tf = transforms.Compose([
            transforms.Resize((sz, sz)),
            transforms.ToTensor(),
            transforms.Normalize(self.MEAN, self.STD),
        ])

    def setup(self):
        full_ds = datasets.ImageFolder(self.data_dir, transform=self.eval_tf)
        self.class_names = full_ds.classes
        targets = full_ds.targets

        train_idx, temp_idx = train_test_split(
            range(len(full_ds)), test_size=0.30,
            stratify=targets, random_state=self.seed,
        )
        temp_targets = [targets[i] for i in temp_idx]
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.50,
            stratify=temp_targets, random_state=self.seed,
        )

        # train_ds dùng augmentation riêng; val/test dùng eval_tf
        train_ds = datasets.ImageFolder(self.data_dir, transform=self.train_tf)
        self.train_dataset = Subset(train_ds, list(train_idx))
        self.val_dataset   = Subset(full_ds,  list(val_idx))
        self.test_dataset  = Subset(full_ds,  list(test_idx))

        print(f'Classes : {self.class_names}')
        print(f'Train   : {len(self.train_dataset)}')
        print(f'Val     : {len(self.val_dataset)}')
        print(f'Test    : {len(self.test_dataset)}')
        return self

    def get_loaders(self, num_workers=2):
        return {
            'train': DataLoader(self.train_dataset, batch_size=self.batch_size,
                                shuffle=True,  num_workers=num_workers, pin_memory=True),
            'val'  : DataLoader(self.val_dataset,   batch_size=self.batch_size,
                                shuffle=False, num_workers=num_workers, pin_memory=True),
            'test' : DataLoader(self.test_dataset,  batch_size=self.batch_size,
                                shuffle=False, num_workers=num_workers, pin_memory=True),
        }

    def show_augmented_samples(self, n=8):
        full_eval  = datasets.ImageFolder(self.data_dir, transform=self.eval_tf)
        full_train = datasets.ImageFolder(self.data_dir, transform=self.train_tf)
        inv = transforms.Normalize(
            [-m/s for m, s in zip(self.MEAN, self.STD)],
            [1/s for s in self.STD],
        )
        fig, axes = plt.subplots(2, n, figsize=(n * 2, 5))
        indices = random.sample(range(len(full_eval)), n)
        for col, idx in enumerate(indices):
            for row, ds in enumerate([full_eval, full_train]):
                t, lbl = ds[idx]
                img = inv(t).permute(1, 2, 0).clamp(0, 1).numpy()
                axes[row, col].imshow(img)
                axes[row, col].set_title(self.class_names[lbl], fontsize=8)
                axes[row, col].axis('off')
        axes[0, 0].set_ylabel('Original',  fontsize=10)
        axes[1, 0].set_ylabel('Augmented', fontsize=10)
        plt.suptitle('Augmentation preview', fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()


dm = SolarPanelDataModule(DATA_DIR, IMG_SIZE, BATCH_SIZE, SEED)
dm.setup()
loaders = dm.get_loaders()
dm.show_augmented_samples(n=6)

## 5. ModelFactory — EfficientNetB4, ResNet50, ViT-B/16

In [ ]:
class ModelFactory:

    @staticmethod
    def build(model_name: str, num_classes: int, freeze_backbone: bool = True) -> nn.Module:
        name = model_name.lower()
        print(f'Building {name.upper()}  freeze_backbone={freeze_backbone}')

        if name == 'efficientnetb4':
            model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
            if freeze_backbone:
                for p in model.features.parameters():
                    p.requires_grad = False
            in_f = model.classifier[1].in_features
            model.classifier = nn.Sequential(
                nn.Dropout(0.4),
                nn.Linear(in_f, 512), nn.BatchNorm1d(512), nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(512, num_classes),
            )

        elif name == 'resnet50':
            model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
            if freeze_backbone:
                # Freeze chỉ layer1/2, giữ layer3/4 trainable từ đầu
                for p in model.layer1.parameters(): p.requires_grad = False
                for p in model.layer2.parameters(): p.requires_grad = False
            in_f = model.fc.in_features
            model.fc = nn.Sequential(
                nn.Linear(in_f, 512), nn.BatchNorm1d(512), nn.ReLU(),
                nn.Dropout(0.4),
                nn.Linear(512, num_classes),
            )

        elif name == 'vit':
            model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
            if freeze_backbone:
                for p in model.encoder.parameters():
                    p.requires_grad = False
            in_f = model.heads.head.in_features
            model.heads.head = nn.Sequential(
                nn.Linear(in_f, 256), nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes),
            )

        else:
            raise ValueError(f'Unknown model: {model_name}')

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f'  Trainable: {trainable:,} / {total:,}  ({trainable/total*100:.1f}%)')
        return model.to(DEVICE)


for n in ['efficientnetb4', 'resnet50', 'vit']:
    m = ModelFactory.build(n, NUM_CLASSES, freeze_backbone=True)
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 6. Trainer — 2-Phase Training + Early Stopping

In [ ]:
class Trainer:

    def __init__(self, model, model_name, loaders, device,
                 models_dir='models', patience=5):
        self.model      = model
        self.model_name = model_name
        self.loaders    = loaders
        self.device     = device
        self.models_dir = models_dir
        self.patience   = patience
        self.history    = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

        # Class-weighted loss để bù imbalance (Snow ít hơn Clean ~3.5x)
        w = torch.tensor(1.0 / np.array(CLASS_COUNTS, dtype=float), dtype=torch.float32)
        w = (w / w.sum() * NUM_CLASSES).to(device)
        self.criterion = nn.CrossEntropyLoss(weight=w)

    def _run_epoch(self, phase: str, optimizer):
        is_train = phase == 'train'
        self.model.train() if is_train else self.model.eval()
        running_loss = correct = total = 0

        with torch.set_grad_enabled(is_train):
            for inputs, labels in tqdm(self.loaders[phase], desc=f'  {phase}', leave=False):
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                if is_train: optimizer.zero_grad()
                outputs = self.model(inputs)
                loss    = self.criterion(outputs, labels)
                if is_train:
                    loss.backward()
                    optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                correct  += (preds == labels).sum().item()
                total    += labels.size(0)

        return running_loss / total, correct / total

    def train(self, num_epochs: int, lr: float, phase_label: str):
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=lr, weight_decay=1e-4,
        )
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr * 0.01)

        best_val_acc = 0.0
        best_wts     = copy.deepcopy(self.model.state_dict())
        no_improve   = 0

        print(f'\n{"="*60}')
        print(f'[{self.model_name.upper()}] {phase_label} — {num_epochs} epochs  lr={lr}')
        print(f'{"="*60}')

        for epoch in range(1, num_epochs + 1):
            t0 = time.time()
            tl, ta = self._run_epoch('train', optimizer)
            vl, va = self._run_epoch('val',   optimizer)
            scheduler.step()

            for k, v in zip(['train_loss','train_acc','val_loss','val_acc'], [tl,ta,vl,va]):
                self.history[k].append(v)

            star = ' *' if va > best_val_acc else ''
            print(f'  [{epoch:02d}/{num_epochs}] '
                  f'Train {tl:.4f}/{ta:.4f}  Val {vl:.4f}/{va:.4f}  '
                  f'{time.time()-t0:.1f}s{star}')

            if va > best_val_acc:
                best_val_acc = va
                best_wts     = copy.deepcopy(self.model.state_dict())
                no_improve   = 0
            else:
                no_improve += 1
                if no_improve >= self.patience:
                    print(f'  Early stopping at epoch {epoch}')
                    break

        self.model.load_state_dict(best_wts)
        print(f'  Best Val Acc: {best_val_acc:.4f}')
        return best_val_acc

    def save(self, suffix=''):
        path = os.path.join(self.models_dir, f'{self.model_name}{suffix}_best.pt')
        torch.save(self.model.state_dict(), path)
        print(f'  Saved -> {path}')
        return path

    def plot_history(self):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        epochs = range(1, len(self.history['train_loss']) + 1)
        for ax, metric in zip(axes, ['loss', 'acc']):
            ax.plot(epochs, self.history[f'train_{metric}'], 'b-o', label=f'Train', ms=4)
            ax.plot(epochs, self.history[f'val_{metric}'],   'r-o', label=f'Val',   ms=4)
            ax.set_title(f'{self.model_name.upper()} — {metric.capitalize()}')
            ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()


print('Trainer ready.')

## 7. Train 3 Models — Phase 1 (Head) + Phase 2 (Fine-tune)

In [ ]:
MODEL_NAMES = ['efficientnetb4', 'resnet50', 'vit']
trainers    = {}

for model_name in MODEL_NAMES:
    print(f'\n{"#"*60}')
    print(f'# TRAIN: {model_name.upper()}')
    print(f'{"#"*60}')

    # Phase 1: freeze backbone, chỉ train head
    model   = ModelFactory.build(model_name, NUM_CLASSES, freeze_backbone=True)
    trainer = Trainer(model, model_name, loaders, DEVICE, MODELS_DIR, patience=5)
    trainer.train(NUM_EPOCHS_PHASE1, LR_PHASE1, 'Phase 1 — Head only')

    # Phase 2: unfreeze toàn bộ, fine-tune với lr nhỏ
    print(f'  Unfreeze all layers...')
    for p in model.parameters(): p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable params: {trainable:,}')

    trainer.train(NUM_EPOCHS_PHASE2, LR_PHASE2, 'Phase 2 — Full fine-tune')
    trainer.save()
    trainer.plot_history()

    trainers[model_name] = trainer
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nTrain xong 3 model.')

## 8. Evaluator — Validation & So sánh

In [ ]:
class Evaluator:

    def __init__(self, device, class_names):
        self.device      = device
        self.class_names = class_names

    def predict(self, model, loader):
        model.eval()
        y_true, y_pred, y_prob = [], [], []
        with torch.no_grad():
            for inputs, labels in tqdm(loader, desc='  Predicting', leave=False):
                outputs = model(inputs.to(self.device))
                probs   = torch.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                y_true.extend(labels.numpy())
                y_pred.extend(preds.cpu().numpy())
                y_prob.extend(probs.cpu().numpy())
        return np.array(y_true), np.array(y_pred), np.array(y_prob)

    def compute_metrics(self, y_true, y_pred):
        return {
            'accuracy' : accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall'   : recall_score(y_true, y_pred,    average='macro', zero_division=0),
            'f1'       : f1_score(y_true, y_pred,        average='macro', zero_division=0),
        }

    def print_report(self, model_name, y_true, y_pred, split='Val'):
        m = self.compute_metrics(y_true, y_pred)
        print(f'\n[{model_name.upper()}] {split} — Acc:{m["accuracy"]:.4f}  '
              f'P:{m["precision"]:.4f}  R:{m["recall"]:.4f}  F1:{m["f1"]:.4f}')
        print(classification_report(y_true, y_pred, target_names=self.class_names, zero_division=0))
        return m

    def plot_confusion_matrix(self, model_name, y_true, y_pred, split='Val'):
        cm      = confusion_matrix(y_true, y_pred)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for ax, data, fmt, title in zip(
            axes, [cm, cm_norm], ['d', '.2f'],
            ['Count', 'Normalized'],
        ):
            sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                        xticklabels=self.class_names, yticklabels=self.class_names)
            ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
            ax.set_title(f'{model_name.upper()} — {title} ({split})')
        plt.tight_layout(); plt.show()


evaluator   = Evaluator(DEVICE, CLASS_NAMES)
val_results = {}

for model_name in MODEL_NAMES:
    print(f'\n--- Validation: {model_name.upper()} ---')
    model = ModelFactory.build(model_name, NUM_CLASSES, freeze_backbone=False)
    model.load_state_dict(torch.load(
        os.path.join(MODELS_DIR, f'{model_name}_best.pt'),
        map_location=DEVICE, weights_only=True,
    ))
    y_true, y_pred, _ = evaluator.predict(model, loaders['val'])
    val_results[model_name] = evaluator.print_report(model_name, y_true, y_pred, 'Val')
    evaluator.plot_confusion_matrix(model_name, y_true, y_pred, 'Val')
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nValidation summary:')
print(pd.DataFrame(val_results).T.round(4).to_string())

## 9. Testing & Classification Report

In [ ]:
test_results = {}

for model_name in MODEL_NAMES:
    print(f'\n--- Testing: {model_name.upper()} ---')
    model = ModelFactory.build(model_name, NUM_CLASSES, freeze_backbone=False)
    model.load_state_dict(torch.load(
        os.path.join(MODELS_DIR, f'{model_name}_best.pt'),
        map_location=DEVICE, weights_only=True,
    ))
    y_true, y_pred, _ = evaluator.predict(model, loaders['test'])
    test_results[model_name] = evaluator.print_report(model_name, y_true, y_pred, 'Test')
    evaluator.plot_confusion_matrix(model_name, y_true, y_pred, 'Test')
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nTest summary:')
print(pd.DataFrame(test_results).T.round(4).to_string())

## 10. Predicted Class Output — Visualize

In [ ]:
class PredictionVisualizer:

    def __init__(self, class_names, device):
        self.class_names = class_names
        self.device      = device
        self.inv = transforms.Normalize(
            [-0.485/0.229, -0.456/0.224, -0.406/0.225],
            [1/0.229, 1/0.224, 1/0.225],
        )

    def _to_img(self, t):
        return self.inv(t).permute(1, 2, 0).clamp(0, 1).numpy()

    def show_predictions(self, model, loader, n_correct=8, n_wrong=8):
        model.eval()
        correct_items, wrong_items = [], []

        with torch.no_grad():
            for inputs, labels in loader:
                outputs = model(inputs.to(self.device))
                confs, preds = torch.max(torch.softmax(outputs, 1), 1)
                for img, lbl, pred, conf in zip(inputs, labels, preds.cpu(), confs.cpu()):
                    item = (img, lbl.item(), pred.item(), conf.item())
                    if pred == lbl and len(correct_items) < n_correct:
                        correct_items.append(item)
                    elif pred != lbl and len(wrong_items) < n_wrong:
                        wrong_items.append(item)
                if len(correct_items) >= n_correct and len(wrong_items) >= n_wrong:
                    break

        for title, items in [('Correct', correct_items), ('Wrong', wrong_items)]:
            if not items: continue
            cols = min(len(items), 8)
            rows = (len(items) + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.8))
            axes = np.array(axes).flatten()
            for ax, (img, true, pred, conf) in zip(axes, items):
                ax.imshow(self._to_img(img))
                color = 'green' if pred == true else 'red'
                ax.set_title(
                    f'T:{self.class_names[true]}\nP:{self.class_names[pred]} {conf:.0%}',
                    fontsize=8, color=color,
                )
                ax.axis('off')
            for ax in axes[len(items):]: ax.axis('off')
            plt.suptitle(f'{title} predictions', fontsize=13, fontweight='bold')
            plt.tight_layout(); plt.show()


best_model_name = max(test_results, key=lambda k: test_results[k]['accuracy'])
print(f'Best model to visualize: {best_model_name.upper()}')

best_model = ModelFactory.build(best_model_name, NUM_CLASSES, freeze_backbone=False)
best_model.load_state_dict(torch.load(
    os.path.join(MODELS_DIR, f'{best_model_name}_best.pt'),
    map_location=DEVICE, weights_only=True,
))

viz = PredictionVisualizer(CLASS_NAMES, DEVICE)
viz.show_predictions(best_model, loaders['test'], n_correct=8, n_wrong=8)

## 11. Best Model Selection

In [ ]:
df_cmp = pd.DataFrame({
    'Model'    : [n.upper() for n in test_results],
    'Accuracy' : [v['accuracy']  for v in test_results.values()],
    'Precision': [v['precision'] for v in test_results.values()],
    'Recall'   : [v['recall']    for v in test_results.values()],
    'F1'       : [v['f1']        for v in test_results.values()],
}).sort_values('Accuracy', ascending=False)

print('FINAL COMPARISON (Test Set):')
print(df_cmp.to_string(index=False))

best_model_name = df_cmp.iloc[0]['Model'].lower()
print(f'\nBest: {best_model_name.upper()}  Acc={df_cmp.iloc[0]["Accuracy"]:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
x    = np.arange(len(df_cmp))
w    = 0.2
cols = ['Accuracy', 'Precision', 'Recall', 'F1']
clrs = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']

for i, (col, clr) in enumerate(zip(cols, clrs)):
    bars = ax.bar(x + i*w, df_cmp[col].astype(float), w, label=col, color=clr, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + w * 1.5); ax.set_xticklabels(df_cmp['Model'])
ax.set_ylim(0.7, 1.05); ax.set_ylabel('Score')
ax.set_title('So sánh 3 Models — Test Set')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 12. Advanced Fine-tuning — Label Smoothing + Mixup + LLRD

In [ ]:
# AdvancedTrainer kế thừa Trainer, override _run_epoch để thêm Mixup
# và dùng train_with_llrd thay train thông thường

class AdvancedTrainer(Trainer):

    def __init__(self, model, model_name, loaders, device,
                 models_dir='models', patience=5,
                 label_smoothing=0.1, mixup_alpha=0.2):
        super().__init__(model, model_name, loaders, device, models_dir, patience)
        self.mixup_alpha = mixup_alpha
        w = torch.tensor(1.0 / np.array(CLASS_COUNTS, dtype=float), dtype=torch.float32)
        w = (w / w.sum() * NUM_CLASSES).to(device)
        self.criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=label_smoothing)
        print(f'  LabelSmoothing={label_smoothing}  MixupAlpha={mixup_alpha}')

    def _mixup_batch(self, inputs, labels):
        lam  = np.random.beta(self.mixup_alpha, self.mixup_alpha)
        idx  = torch.randperm(inputs.size(0)).to(inputs.device)
        return lam * inputs + (1 - lam) * inputs[idx], labels, labels[idx], lam

    def _run_epoch(self, phase, optimizer):
        is_train = phase == 'train'
        self.model.train() if is_train else self.model.eval()
        running_loss = correct = total = 0

        with torch.set_grad_enabled(is_train):
            for inputs, labels in tqdm(self.loaders[phase], desc=f'  {phase}', leave=False):
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                if is_train and self.mixup_alpha > 0:
                    inputs, la, lb, lam = self._mixup_batch(inputs, labels)
                    optimizer.zero_grad()
                    outputs = self.model(inputs)
                    loss    = lam * self.criterion(outputs, la) + (1 - lam) * self.criterion(outputs, lb)
                    loss.backward(); optimizer.step()
                    ref_labels = la
                else:
                    if is_train: optimizer.zero_grad()
                    outputs = self.model(inputs)
                    loss    = self.criterion(outputs, labels)
                    if is_train: loss.backward(); optimizer.step()
                    ref_labels = labels

                running_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                correct  += (preds == ref_labels).sum().item()
                total    += inputs.size(0)

        return running_loss / total, correct / total

    def train_with_llrd(self, num_epochs, base_lr, decay_factor=0.1):
        # LLRD: backbone lr thấp, head lr cao
        head_names = ['classifier', 'fc', 'heads']
        backbone_p, head_p = [], []
        for name, p in self.model.named_parameters():
            (head_p if any(h in name for h in head_names) else backbone_p).append(p)

        optimizer = optim.AdamW([
            {'params': backbone_p, 'lr': base_lr * decay_factor},
            {'params': head_p,     'lr': base_lr},
        ], weight_decay=1e-4)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=base_lr * 0.001)

        best_val_acc = 0.0
        best_wts     = copy.deepcopy(self.model.state_dict())
        no_improve   = 0

        print(f'\n{"="*60}')
        print(f'[{self.model_name.upper()}] LLRD+Mixup+LabelSmooth')
        print(f'  Backbone LR: {base_lr*decay_factor:.2e}  Head LR: {base_lr:.2e}')
        print(f'{"="*60}')

        for epoch in range(1, num_epochs + 1):
            t0 = time.time()
            tl, ta = self._run_epoch('train', optimizer)
            vl, va = self._run_epoch('val',   optimizer)
            scheduler.step()

            for k, v in zip(['train_loss','train_acc','val_loss','val_acc'], [tl,ta,vl,va]):
                self.history[k].append(v)

            star = ' *' if va > best_val_acc else ''
            print(f'  [{epoch:02d}/{num_epochs}] '
                  f'Train {tl:.4f}/{ta:.4f}  Val {vl:.4f}/{va:.4f}  '
                  f'{time.time()-t0:.1f}s{star}')

            if va > best_val_acc:
                best_val_acc = va
                best_wts     = copy.deepcopy(self.model.state_dict())
                no_improve   = 0
            else:
                no_improve += 1
                if no_improve >= self.patience:
                    print(f'  Early stopping at epoch {epoch}')
                    break

        self.model.load_state_dict(best_wts)
        print(f'  Best Val Acc: {best_val_acc:.4f}')
        return best_val_acc


print(f'Improving: {best_model_name.upper()}')

improved_model = ModelFactory.build(best_model_name, NUM_CLASSES, freeze_backbone=False)
improved_model.load_state_dict(torch.load(
    os.path.join(MODELS_DIR, f'{best_model_name}_best.pt'),
    map_location=DEVICE, weights_only=True,
))

adv_trainer = AdvancedTrainer(
    improved_model, f'{best_model_name}_improved',
    loaders, DEVICE, MODELS_DIR, patience=6,
    label_smoothing=0.1, mixup_alpha=0.2,
)
adv_trainer.train_with_llrd(num_epochs=15, base_lr=5e-5, decay_factor=0.1)
adv_trainer.save()
adv_trainer.plot_history()

## 13. Test-Time Augmentation (TTA)

In [ ]:
# TTA: inference 5 lần với transform khác nhau, average softmax probability

class TTAEvaluator:

    _NORM = {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]}

    TTA_TRANSFORMS = [
        transforms.Compose([transforms.Resize((224,224)),
                             transforms.ToTensor(), transforms.Normalize(**_NORM)]),
        transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=1.0),
                             transforms.ToTensor(), transforms.Normalize(**_NORM)]),
        transforms.Compose([transforms.Resize((256,256)), transforms.CenterCrop(224),
                             transforms.ToTensor(), transforms.Normalize(**_NORM)]),
        transforms.Compose([transforms.Resize((224,224)),
                             transforms.ColorJitter(brightness=0.2, contrast=0.2),
                             transforms.ToTensor(), transforms.Normalize(**_NORM)]),
        transforms.Compose([transforms.Resize((224,224)), transforms.RandomRotation(10),
                             transforms.ToTensor(), transforms.Normalize(**_NORM)]),
    ]

    def __init__(self, data_dir, test_indices, device, class_names, batch_size=32):
        self.data_dir     = data_dir
        self.test_indices = test_indices
        self.device       = device
        self.class_names  = class_names
        self.batch_size   = batch_size

    def predict_tta(self, model):
        model.eval()
        all_probs = None
        y_true    = None

        for i, tf in enumerate(self.TTA_TRANSFORMS):
            print(f'  TTA pass {i+1}/{len(self.TTA_TRANSFORMS)}')
            subset = Subset(datasets.ImageFolder(self.data_dir, transform=tf), self.test_indices)
            loader = DataLoader(subset, batch_size=self.batch_size, shuffle=False, num_workers=2)

            probs_i, labels_i = [], []
            with torch.no_grad():
                for inputs, labels in loader:
                    out = model(inputs.to(self.device))
                    probs_i.extend(torch.softmax(out, 1).cpu().numpy())
                    if i == 0: labels_i.extend(labels.numpy())

            probs_arr = np.array(probs_i)
            all_probs = probs_arr if all_probs is None else all_probs + probs_arr
            if i == 0: y_true = np.array(labels_i)

        y_pred = np.argmax(all_probs / len(self.TTA_TRANSFORMS), axis=1)
        return y_true, y_pred


improved_path  = os.path.join(MODELS_DIR, f'{best_model_name}_improved_best.pt')
improved_model = ModelFactory.build(best_model_name, NUM_CLASSES, freeze_backbone=False)
improved_model.load_state_dict(torch.load(improved_path, map_location=DEVICE, weights_only=True))

tta_eval = TTAEvaluator(DATA_DIR, list(dm.test_dataset.indices), DEVICE, CLASS_NAMES, BATCH_SIZE)

print(f'TTA inference on {len(dm.test_dataset)} test images...')
y_true_tta, y_pred_tta = tta_eval.predict_tta(improved_model)

tta_acc = accuracy_score(y_true_tta, y_pred_tta)
tta_f1  = f1_score(y_true_tta, y_pred_tta, average='macro', zero_division=0)
print(f'\nTTA Result  Acc={tta_acc:.4f}  F1={tta_f1:.4f}')
print(classification_report(y_true_tta, y_pred_tta, target_names=CLASS_NAMES, zero_division=0))

## 14. Tổng kết Experiment

In [ ]:
improved_model_reloaded = ModelFactory.build(best_model_name, NUM_CLASSES, freeze_backbone=False)
improved_model_reloaded.load_state_dict(
    torch.load(improved_path, map_location=DEVICE, weights_only=True)
)
y_true_imp, y_pred_imp, _ = evaluator.predict(improved_model_reloaded, loaders['test'])
m_imp = evaluator.compute_metrics(y_true_imp, y_pred_imp)

rows = []
for name, m in test_results.items():
    rows.append({'Model': name.upper(), 'Variant': 'Baseline',
                 'Accuracy': round(m['accuracy'],4), 'Precision': round(m['precision'],4),
                 'Recall': round(m['recall'],4),     'F1': round(m['f1'],4)})
rows.append({'Model': best_model_name.upper(), 'Variant': 'Improved',
             'Accuracy': round(m_imp['accuracy'],4), 'Precision': round(m_imp['precision'],4),
             'Recall': round(m_imp['recall'],4),     'F1': round(m_imp['f1'],4)})
rows.append({'Model': f'{best_model_name.upper()}+TTA', 'Variant': 'Improved+TTA',
             'Accuracy': round(tta_acc,4), 'Precision': '-',
             'Recall': '-',                'F1': round(tta_f1,4)})

df_summary = pd.DataFrame(rows)
print('='*80)
print('SOLAR PANEL CLASSIFICATION — Final Summary')
print('='*80)
print(df_summary.to_string(index=False))
print('='*80)
print(f'Best: {best_model_name.upper()} (Improved+TTA)  Acc={tta_acc:.4f}  F1={tta_f1:.4f}')

print('\nSaved models:')
for f in sorted(os.listdir(MODELS_DIR)):
    if f.endswith('.pt'):
        size = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1e6
        print(f'  {f:<50} {size:.1f} MB')